# Kaggriculture Tutorial

Farm your way to market dominance! Players harvest produce and animal products to sell in a dynamic market.

## Game Mechanics 
 
- **Farmer**: takes one action per turn over a season with 30 days and 24 turns per day (720 total).
- **Crops and animals**: each have their own seed cost, time to first yield, and total payout. 
- **Daily care**: plants need watering every day or they turn to weeds, animals need feeding or they escape.
- **Market prices**: move with supply, selling a product pushes its price down, and crops vary in how hard they crash from a glut.
- **Town shops** unlock over the season and steadily buy products, lifting prices over time.
- **Farm hands** can be hired for the day, with increasing costs for each hire per day. 
- **Farm expansion**: start with one quadrant of land and can buy the other three for an escalating fee.
- **Shed**: holds harvested goods but caps at 100 items, anything past that is discarded at end of day.
 - **Win condition**: whoever has the most money in the bank at the end of the season wins.

In [ ]:

from kaggle_environments import make

env = make("kaggriculture", debug=True)
print(f"Environment: {env.name} v{env.version}")
print(f"Players: {env.specification.agents}")
print(f"Max steps: {env.configuration.episodeSteps}")

## Understanding the Observation

Each turn your agent receives an observation with:

- **`player`** — your player id (`0` or `1`).
- **`day`** / **`hour`** — the current in-game day (0-indexed) and turn within that day (0-indexed; there are `turnsPerDay` turns per day).
- **`farms`** — a list of both players' public farm state, indexed by player id. Each farm has:
    - `money` — current bank balance
    - `tiles` — a `boardSize × boardSize` grid indexed `tiles[y][x]`; each cell is `None` (empty), `"LOCKED"` (unowned quadrant), or a dict describing a `PLANT`, `WEED`, `COOP`, or `PASTURE` (see below)
    - `farmer` — `[x, y]` position of your main farmer
    - `hands` — `[x, y]` positions of any hired hands active today
    - `unlocked_quadrants` — subset of `["NW", "NE", "SW", "SE"]`
    - `hires_today` — number of hires already made today (drives the next `HIRE` price)
- **`market`** (shared) — `inventory` and current `prices` per product (`WHEAT, CARROT, TOMATO, STRAWBERRY, MELON, EGG, MILK, WOOL, FERTILIZER`).
- **`town`** (shared) — `unlocked_shops`, the list of shops currently generating per-tick demand.
- **`private`** — your own hidden state (not visible to your opponent):
  - `shed` — counts of every product and animal (`GOOSE`, `COW`, `SHEEP`) stored in your shed
  - `seeds` — seed counts per crop
  - `inventories` — per-unit carried inventories; `[0]` is your main farmer, `[1..]` are today's hired hands in order

Tile dicts come in a few shapes:

- **Plant**: `{kind: "PLANT", crop, planted_day, watered_today, consecutive_unwatered, yield_units, max_lifespan_step, fertilized_until_day}` — `consecutive_unwatered >= 2` turns the tile to a weed at end of day.
- **Weed**: `{kind: "WEED"}` — must be `DIG`-ed before the tile is usable again.
- **Coop / Pasture** (empty): `{kind: "COOP" | "PASTURE"}`.
- **Coop / Pasture** (occupied): adds `animal, placed_day, yield_units, fed_today, consecutive_unfed, cared_today, fertilizer_available, pending_care_bonus`. `consecutive_unfed >= 2` means the animal escapes.

Your agent returns a dict of actions for the farmer, any farm hands, and the market: `{"farmer": 'PASS', "hands": [], "market": []}`


In [ ]:
# Run a quick game to see what the observation looks like
env = make("kaggriculture", debug=True)
env.run(["random", "random"])

# Peek at the initial observation
obs = env.steps[1][0].observation  # step 1 = first action step
items = obs.market.prices.keys()
print(f"Player: {obs.player}")
print(f"Player {obs.player}'s Unlocked Farm Areas: {obs.farms[obs.player].unlocked_quadrants}")
for i in items:
    print(f"{i} Price: {obs.market.prices[i]}")

In [ ]:
from kaggle_environments import make


In [ ]:
# ============================================================
# CROP DATA
# Static information provided by the Kaggriculture competition.
#
# These values describe how each crop behaves.
# Current market prices are NOT stored here because they
# change during the game and are provided by obs["market"].
# ============================================================

CROP_DATA = {

    # Wheat is a one-time crop.
    # Without fertilizer, its maximum yield is 4.
    # With fertilizer, its maximum yield is 6.
    "WHEAT": {
        "yield_type": "ONE_TIME",
        "seed_cost": 10,
        "base_price": 25,
        "first_yield_day": 2,
        "max_yield_day": 4,
        "max_yield": 6,
        "unfertilized_max_yield": 4,
        "yield_per_tile_day": 0.80,
    },

    # Carrot is a one-time crop.
    "CARROT": {
        "yield_type": "ONE_TIME",
        "seed_cost": 20,
        "base_price": 35,
        "first_yield_day": 2,
        "max_yield_day": 3,
        "max_yield": 4,
        "unfertilized_max_yield": 3,
        "yield_per_tile_day": 0.75,
    },

    # Tomato is an ongoing crop with four scheduled yields.
    "TOMATO": {
        "yield_type": "ONGOING",
        "seed_cost": 50,
        "base_price": 60,
        "first_yield_day": 8,
        "max_yield_day": 11,
        "max_yield": 4,
        "subsequent_yield_interval": 1,
        "yield_per_tile_day": 0.33,
    },

    # Strawberry is an ongoing crop with four scheduled yields.
    "STRAWBERRY": {
        "yield_type": "ONGOING",
        "seed_cost": 100,
        "base_price": 120,
        "first_yield_day": 10,
        "max_yield_day": 16,
        "max_yield": 4,
        "subsequent_yield_interval": 2,
        "yield_per_tile_day": 0.24,
    },

    # Melon is a one-time crop with a high base market price.
    "MELON": {
        "yield_type": "ONE_TIME",
        "seed_cost": 80,
        "base_price": 250,
        "first_yield_day": 10,
        "max_yield_day": 10,
        "max_yield": 6,
        "unfertilized_max_yield": 6,
        "yield_per_tile_day": 0.55,
    },
}


# ============================================================
# ANIMAL DATA
# Static information provided by the Kaggriculture competition.
#
# Each animal produces a specific product over time.
# Animal production continues as long as the animal survives.
# ============================================================

ANIMAL_DATA = {

    # Geese live on a COOP and produce eggs every day
    # after the initial production delay.
    "GOOSE": {
        "product": "EGG",
        "purchase_cost": 300,
        "base_price": 50,
        "first_yield_day": 4,
        "yield_interval_days": 1,
        "max_held": 4,
        "structure": "COOP",
        "yield_per_tile_day": 1.00,
    },

    # Cows live on a PASTURE and produce milk every two days.
    "COW": {
        "product": "MILK",
        "purchase_cost": 400,
        "base_price": 160,
        "first_yield_day": 8,
        "yield_interval_days": 2,
        "max_held": 6,
        "structure": "PASTURE",
        "yield_per_tile_day": 0.50,
    },

    # Sheep live on a PASTURE and produce wool every three days.
    "SHEEP": {
        "product": "WOOL",
        "purchase_cost": 500,
        "base_price": 200,
        "first_yield_day": 6,
        "yield_interval_days": 3,
        "max_held": 6,
        "structure": "PASTURE",
        "yield_per_tile_day": 0.33,
    },
}


# ============================================================
# PRODUCT DATA
# Describes where each market product comes from.
#
# The current market price is NOT stored here.
# It must always be read from:
# obs["market"]["prices"][product]
# ============================================================

PRODUCT_DATA = {

    # Crop products
    "WHEAT": {
        "source_type": "CROP",
        "source": "WHEAT",
    },

    "CARROT": {
        "source_type": "CROP",
        "source": "CARROT",
    },

    "TOMATO": {
        "source_type": "CROP",
        "source": "TOMATO",
    },

    "STRAWBERRY": {
        "source_type": "CROP",
        "source": "STRAWBERRY",
    },

    "MELON": {
        "source_type": "CROP",
        "source": "MELON",
    },

    # Animal products
    "EGG": {
        "source_type": "ANIMAL",
        "source": "GOOSE",
    },

    "MILK": {
        "source_type": "ANIMAL",
        "source": "COW",
    },

    "WOOL": {
        "source_type": "ANIMAL",
        "source": "SHEEP",
    },

    # Fertilizer is produced by all surviving animals.
    "FERTILIZER": {
        "source_type": "ANIMAL_BYPRODUCT",
        "source": "ALL_ANIMALS",
    },
}

In [ ]:
def is_crop_ready_to_harvest(obs,tile):
    """
    Determine whether a crop has reached its configured harvest point.

    The decision is based on:
    - the tile being a plant;
    - the crop having available yield;
    - the crop's age;
    - the crop's configured max_yield_day.
    """

    if tile is None:
        return False

    if tile.get("kind") != "PLANT":
        return False
    crop_name = tile["crop"]
    crop = CROP_DATA[crop_name]

    age = obs["day"] - tile["planted_day"]

    min_harvest_yield = max(
        1,
        crop["unfertilized_max_yield"] - 1
    )

    return (tile.get("yield_units", 0) >= min_harvest_yield 
            and age >= crop["max_yield_day"]
    )

In [ ]:
def get_harvest_priority(obs, tile, x, y, fx, fy):

    if tile is None or tile.get("kind") != "PLANT":
        return None
    
    crop = CROP_DATA[tile["crop"]]
    yield_units = tile.get("yield_units", 0)
    max_yield = crop["unfertilized_max_yield"]  # use max_yield sem fertilizante

    if yield_units <= 0:
        return None

    steps_to_decay = (
        tile["max_lifespan_step"] - obs["step"]
    )

    distance = abs(x - fx) + abs(y - fy)

    # Quanto tempo sobra depois de chegar à planta
    slack = steps_to_decay - distance

    # ---------------------------------------------------------
    # YIELD MÁXIMO
    # ---------------------------------------------------------
    if yield_units >= max_yield:

        # Está perto demais do decay.
        #if slack <= 2:
            return 0

        # Já está no máximo, mas ainda pode esperar.
       #return None

    # ---------------------------------------------------------
    # YIELD 2
    # ---------------------------------------------------------
    if yield_units >= max_yield - 1:

        # Se ainda há tempo razoável para continuar crescendo,
        # não colher ainda.
        if slack > 2:
            return None

        # Está ficando perigoso esperar.
        return 0

    # ---------------------------------------------------------
    # YIELD 1
    # ---------------------------------------------------------
    return None

In [ ]:
# ============================================================
# CURRENT EXPERIMENT
# ============================================================
# Crop currently being used by the experimental agent.
# Changing this value will allow the same agent logic
# to be tested with different crops later.

CROP = "MELON"

# Get the static information for the selected crop.
crop = CROP_DATA[CROP]

print(f"Selected crop: {CROP}")
print(f"Seed cost: {crop['seed_cost']}")
print(f"Base price: {crop['base_price']}")
print(f"First yield day: {crop['first_yield_day']}")
print(f"Max yield day: {crop['max_yield_day']}")
print(f"Max yield: {crop['max_yield']}")

In [ ]:
def get_farmer_tile(obs):
    player = obs["player"]
    farm = obs["farms"][player]

    x, y = farm["farmer"]

    return farm["tiles"][y][x]

In [ ]:
# Returns one movement action that reduces the Manhattan distance
# between the farmer's current position and the target position.
# The function moves horizontally first, then vertically.
# Returns None when the farmer is already at the target.
'''def step_toward(fx, fy, tx, ty):
    if fx > tx:
        return "WEST"
    if fx < tx:
        return "EAST"
    if fy > ty:
        return "NORTH"
    if fy < ty:
        return "SOUTH"
    return None'''
def step_toward(x, y, tx, ty, first_axis="x"):
    """
    Retorna o próximo movimento em direção ao alvo.

    first_axis:
        "x" -> tenta X primeiro
        "y" -> tenta Y primeiro
    """

    dx = tx - x
    dy = ty - y

    if first_axis == "x":

        if dx > 0:
            return "EAST"
        if dx < 0:
            return "WEST"

        if dy > 0:
            return "SOUTH"
        if dy < 0:
            return "NORTH"

    else:  # first_axis == "y"

        if dy > 0:
            return "SOUTH"
        if dy < 0:
            return "NORTH"

        if dx > 0:
            return "EAST"
        if dx < 0:
            return "WEST"

    return None


In [ ]:
def find_nearby_harvest(obs, x, y, max_distance=6):
    farm = obs["farms"][obs["player"]]

    candidates = []

    for ty, row in enumerate(farm["tiles"]):
        for tx, tile in enumerate(row):

            if not isinstance(tile, dict):
                continue

            if tile.get("kind") != "PLANT":
                continue

            harvest_priority = get_harvest_priority(
                obs,
                tile,
                tx,
                ty,
                x,
                y
            )

            if harvest_priority is None:
                continue

            distance = abs(tx - x) + abs(ty - y)

            if distance <= max_distance:
                candidates.append(
                    (
                        harvest_priority,
                        distance,
                        tx,
                        ty
                    )
                )

    if not candidates:
        return None

    candidates.sort()

    _, _, tx, ty = candidates[0]

    return (tx, ty)

In [ ]:
def find_final_harvest_target(obs):
    """
    Encontra qualquer crop ainda plantado para colheita final.
    Ignora completamente yield, prioridade e tempo de crescimento.
    """

    player = obs["player"]
    farm = obs["farms"][player]

    fx, fy = farm["farmer"]

    candidates = []

    for y, row in enumerate(farm["tiles"]):
        for x, tile in enumerate(row):

            if not isinstance(tile, dict):
                continue

            if tile.get("kind") != "PLANT":
                continue

            distance = abs(x - fx) + abs(y - fy)

            candidates.append(
                (x, y, "harvest", distance)
            )

    if not candidates:
        return None

    # Pega o crop mais próximo
    candidates.sort(key=lambda c: c[3])

    return candidates[0][:3]

In [ ]:
# Finds the best available farm task based on the current observation.
# Tasks are prioritized as harvest, water, and plant.
# Within the same task priority, the nearest tile is selected
# using Manhattan distance.
# The target is recalculated from the current state each turn.
def find_target(obs, have_seed):
    player = obs["player"]
    farm = obs["farms"][player]

    fx, fy = farm["farmer"]
    candidates_farmer = []

    for y, row in enumerate(farm["tiles"]):
        for x, tile in enumerate(row):
            # -------------------------------------------------
            # TILE VAZIO → PLANT
            # -------------------------------------------------
            if tile is None:
                if have_seed:
                                                   
                    days_remaining = 29 - obs["day"]

                    # Para carrot, precisamos de 2 dias
                    # para chegar ao yield 2.
                    min_days_to_yield = crop["first_yield_day"]

                    if days_remaining >= min_days_to_yield:
                        candidates_farmer.append((x, y, "plant")
                        )
                continue
            
            if not isinstance(tile, dict):
                continue
            if tile.get("kind") == "WEED":
                candidates_farmer.append((x, y, "dig"))
                continue
            if tile.get("kind") != "PLANT":
                continue
            # -------------------------------------------------
            # HARVEST
            # -------------------------------------------------
            harvest_priority = get_harvest_priority(
                obs,
                tile,
                x,
                y,
                fx,
                fy
            )

            if harvest_priority is not None:
                candidates_farmer.append(
                    (x, y, "harvest",harvest_priority)
                )

            # A crop needs to be watered today.
            if (not tile["watered_today"]) and tile.get("kind") == "PLANT":
                candidates_farmer.append((x, y, "water"))

            # The crop is ready to harvest.
            #if is_crop_ready_to_harvest(obs, tile):
             #   candidates_farmer.append((x, y, "harvest"))

    if not candidates_farmer:
        return None

    priority = {
        "water": 1,
        "dig": 1,
        "plant": 1,
        "harvest": 0
        
    }

    candidates_farmer.sort(
        key=lambda c: (
            priority[c[2]],
            c[3] if c[2] == "harvest" else 999,
            0 if c[0] == 4 else 1,
            abs(c[0] - fx) + abs(c[1] - fy),
        )
    )
    '''    debug_targets.append({
        "day": obs["day"],
        "hour": obs["hour"],
        "seed": have_seed,
        "candidates": candidates_farmer.copy(),
    })'''
    best = candidates_farmer[0]
    return best[:3]

In [ ]:
def find_hand_target(obs, hand_idx, have_seed, excluded_targets=None):
    if excluded_targets is None:
        excluded_targets = set()
    player = obs["player"]
    farm = obs["farms"][player]

    # ---------------------------------------------------------
    # Posição do hand
    # ---------------------------------------------------------
    hx, hy = farm["hands"][hand_idx]

    # Inventário:
    # inventories[0] = farmer
    # inventories[1] = hand 1
    #inventory = obs["private"]["inventories"][hand_idx + 1]

    # ---------------------------------------------------------
    # Procurar tarefas
    # ---------------------------------------------------------
    candidates = []

    for y, row in enumerate(farm["tiles"]):
        for x, tile in enumerate(row):

            if (x, y) in excluded_targets:
                continue
            # -------------------------------------------------
            # PLANT
            # -------------------------------------------------
            if tile is None:
                if have_seed:
                    candidates.append((x, y, "plant"))
                    
            if not isinstance(tile, dict):
                continue

            if tile.get("kind") == "WEED":
                candidates.append((x, y, "dig"))
                continue
            if tile.get("kind") != "PLANT":
                continue
            # -------------------------------------------------
            # WATER
            # -------------------------------------------------
            # A crop needs to be watered today.
            if (not tile["watered_today"]) and tile.get("kind") == "PLANT":
                # Não deixa o hand regar a mesma planta
                # que o farmer já escolheu.
                candidates.append((x, y, "water"))

            # -------------------------------------------------
            # HARVEST
            # -------------------------------------------------
            harvest_priority = get_harvest_priority(
                obs,
                tile,
                x,
                y,
                hx,
                hy
            )

            if harvest_priority is not None:
                candidates.append(
                    (x, y, "harvest",harvest_priority)
                )

    if not candidates:
        return None

    # ---------------------------------------------------------
    # Prioridade do HAND
    #
    # WATER   = 1
    # HARVEST = 2
    # PLANT   = 4
    # DIG     = 3
    #
    # DROP já é tratado antes.
    # ---------------------------------------------------------
    priority = {
        "water": 1,
        "plant": 1,
        "dig": 2,
        "harvest": 0
        
    }

    candidates.sort(
        key=lambda c: (
            priority[c[2]],
            c[3] if c[2] == "harvest" else 999,
            0 if c[1] == 4 else 1,
            abs(c[0] - hx) + abs(c[1] - hy),
        )
    )

    return candidates[0]

In [ ]:
def get_sell_actions(obs):

    shed = obs["private"]["shed"]
    actions = []

    for product, quantity in shed.items():

        if quantity <= 0:
            continue
        elif quantity >= 5:
        # regras específicas de venda
            actions.append(["SELL", product, 5])
        else:
            actions.append(["SELL",product,quantity])

    return actions

In [ ]:
def save_action(obs, action):
    farm = obs["farms"][obs["player"]]

    action_debug.append({
        "step": obs["step"],
        "day": obs["day"],
        "hour": obs["hour"],
        "farmer_pos": tuple(farm["farmer"]),
        "hands_pos": [tuple(h) for h in farm["hands"]],
        "farmer": action["farmer"],
        "hands": action["hands"],
        "market": action["market"],
    })

    return action

In [ ]:
def get_crop(obs):
    return "MELON" if obs["day"] <= 20 else "CARROT"

In [ ]:
def agent(obs):

    '''observations.append(obs)
    try:
        # Toda a sua lógica principal entra aqui
        action = my_main_logic(obs)
        return action
    except Exception as e:
        print(f"Erro capturado no Step: {e}")
         Retorna uma ação neutra válida no formato esperado pelo Kaggriculture
        return {"farmer": "PASS", "hands": [], "market": []}'''
    # ---------------------------------------------------------
    # ESTADO
    # ---------------------------------------------------------
    CROP = get_crop(obs)
    crop = CROP_DATA[CROP]
    player = obs["player"]
    farm = obs["farms"][player]

    fx, fy = farm["farmer"]
    hands = farm["hands"]

    seeds = obs["private"]["seeds"].get(CROP, 0)
    shed = obs["private"]["shed"].get(CROP, 0)

    farmer_inventory = obs["private"]["inventories"][0]
    inventory = farmer_inventory.get(CROP, 0)

    # ---------------------------------------------------------
    # DEBUG
    # ---------------------------------------------------------
    plant_debug.append({
        "step": obs["step"],
        "day": obs["day"],
        "hour": obs["hour"],
        "plants": [
            {
                "pos": (x, y),
                "yield": tile.get("yield_units", 0),
                "planted_day": tile.get("planted_day"),
                "lifespan_step": tile.get("max_lifespan_step"),
                "steps_to_decay": (
                    tile.get("max_lifespan_step", 0)
                    - obs["step"]
                ),
            }
            for y, row in enumerate(farm["tiles"])
            for x, tile in enumerate(row)
            if isinstance(tile, dict)
            and tile.get("kind") == "PLANT"
        ]
    })
    money_debug.append({
        
        "step": obs["step"],
        "day": obs["day"],
        "hour": obs["hour"],
        "money": farm["money"],
        "seed": seeds,
        "inventory": inventory,
        "shed": shed,

        "active_crops": sum(
            1
            for row in farm["tiles"]
            for tile in row
            if isinstance(tile, dict)
            and tile.get("kind") == "PLANT"
        ),

        "hands": len(hands),
    })

    sell_debug.append({
        "day": obs["day"],
        "hour": obs["hour"],
        "money": farm["money"],
        "shed": shed,
        "seeds": seeds,
        "inventory": inventory,
    })

    # ---------------------------------------------------------
    # MERCADO
    # ---------------------------------------------------------

    market_actions = []

    # Contrata 2 hand por dia
    HANDS_PER_DAY = 1
    max_yield_day = crop["max_yield_day"]
    if (
        obs["day"] > 2
        and (obs["day"] ) % max_yield_day == 0
    ):
        HANDS_PER_DAY = 2
    if obs["hour"] == 0 and obs["day"]<29:
        # Contrata 2 hand por dia a partir do dia 9
        #if obs["day"] >= 29 :#and obs["day"] < 28:
         #   HANDS_PER_DAY = 2

        hires_needed = HANDS_PER_DAY - farm["hires_today"]

        for _ in range(max(0, hires_needed)):
            market_actions.append(["HIRE"])
    # ---------------------------------------------------------
    # CONTROLE DE SEMENTES
    # ---------------------------------------------------------

    MAX_CROPS = 25
    MAX_SEEDS = 4

    active_crops = sum(
        1
        for row in farm["tiles"]
        for tile in row
        if (
            isinstance(tile, dict)
            and tile.get("kind") == "PLANT"
        )
    )

    # Compra apenas quando existe capacidade real
    # de colocar a semente no campo.
                    
    days_remaining = 29 - obs["day"]
    # Para carrot, precisamos de 2 dias
    # para chegar ao yield 2.
    min_days_to_yield = crop["first_yield_day"]
    if (
        active_crops + seeds < MAX_CROPS
        and seeds < MAX_SEEDS 
        and days_remaining >= min_days_to_yield
        and (obs["day"] < 27)
    ):
        market_actions.append(
            ["BUY_SEED", CROP, 2]
        )

    # ---------------------------------------------------------
    # VENDA
    #
    # IMPORTANTE:
    # vender NÃO deve interromper o farmer.
    # ---------------------------------------------------------
    sell_action = get_sell_actions(obs)
    market_actions.extend(sell_action)
    '''if shed <= 5 or obs["day"]>28:
        market_actions.append(
            ["SELL", CROP, shed]
        )
    else:
        market_actions.append(
            ["SELL", CROP, 5]
        )'''

    # ---------------------------------------------------------
    # TARGET DO FARMER
    # ---------------------------------------------------------

    have_seed = seeds > 0

    if obs["day"] >= 29 and active_crops > 0:
        target = find_final_harvest_target(obs)
    else:
        target = find_target(
            obs,
            have_seed
        )
    farmer_target = None

    if target is not None:
        farmer_target = (target[0], target[1])
    elif (obs["day"]==0 and obs["hour"]== 0):
            return save_action(obs,{
            "farmer": ["NORTH"],
            "hands": [],
            "market": market_actions,
        })
    # ---------------------------------------------------------
    # TARGET DO FARM HAND
    # ---------------------------------------------------------
    # ---------------------------------------------------------
    # TARGET DOS FARM HANDS
    # ---------------------------------------------------------

    hands_actions = []

    # O alvo do farmer fica reservado.
    reserved_targets = set()

    if farmer_target is not None:
        reserved_targets.add(farmer_target)

    # Se o farmer vai plantar agora, uma semente já está comprometida.
    farmer_will_plant = (
        target is not None
        and target[2] == "plant"
        and step_toward(fx, fy, target[0], target[1]) is None
    )

    seeds_available_for_hands = seeds

    if farmer_will_plant:
        seeds_available_for_hands -= 1

    # Cria uma ação para cada hand contratado.
    for hand_idx, (hx, hy) in enumerate(hands):

        hand_target = find_hand_target(
            obs,
            hand_idx=hand_idx,
            have_seed=seeds_available_for_hands > 0,
            excluded_targets=reserved_targets,
        )

        if hand_target is None:
            hand_action = ["PASS"]

        else:
            tx, ty, purpose ,*rest = hand_target

            move = step_toward(hx, hy, tx, ty,first_axis="y")

            if move is not None:
                hand_action = [move]
                

            elif purpose == "water":
                hand_action = ["WATER"]

            elif purpose == "plant":
                hand_action = ["PLANT", CROP]

            elif purpose == "harvest":
                hand_action = ["HARVEST"]

            elif purpose == "dig":
                hand_action = ["DIG"]
            else:
                hand_action = ["PASS"]

            # Reserva o alvo para o próximo hand.
            reserved_targets.add((tx, ty))

            # Não deixa dois hands tentarem plantar com a mesma semente.
            if hand_action[0] == "PLANT":
                seeds_available_for_hands -= 1
        hands_actions.append(hand_action)

    # ---------------------------------------------------------
    # FINAL DA PARTIDA — DIA 29
    # ---------------------------------------------------------

    shed_positions = {
        (4, 4),
        (4, 5),
        (5, 4),
        (5, 5),
    }

    FINAL_DAY_RETURN_HOUR = 20

    if obs["day"] == 29 and obs["hour"] >= FINAL_DAY_RETURN_HOUR:

        if inventory > 0 and active_crops == 0:

            # Já está no shed
            if (fx, fy) in shed_positions:

                return save_action(obs, {
                    "farmer": ["DROP"],
                    "hands": hands_actions,
                    "market": market_actions,
                })

            # Vai para o shed
            target_shed = min(
                shed_positions,
                key=lambda p:
                    abs(p[0] - fx) +
                    abs(p[1] - fy)
            )

            move = step_toward(
                fx,
                fy,
                target_shed[0],
                target_shed[1]
            )

            if move is not None:

                return save_action(obs, {
                    "farmer": [move],
                    "hands": hands_actions,
                    "market": market_actions,
                })
                
        # ---------------------------------------------------------
        # EXECUTAR TARGET DO FARMER
        # ---------------------------------------------------------

    if target is not None:

        tx, ty, purpose,*rest = target

        move = step_toward(
            fx,
            fy,
            tx,
            ty
        )

        if move is not None:
            return save_action(obs,{
            "farmer": [move],
            "hands": hands_actions,
            "market": market_actions,
            })

        # ---------------------------------------------
        # Chegou ao alvo
        # ---------------------------------------------

        if purpose == "plant":

            return save_action(obs,{
                "farmer": ["PLANT", CROP],
                "hands": hands_actions,
                "market": market_actions,
            })

        if purpose == "water":

            return save_action(obs, {
                "farmer": ["WATER"],
                "hands": hands_actions,
                "market": market_actions,
            })

        if purpose == "harvest":

            return save_action(obs, {
                "farmer": ["HARVEST"],
                "hands": hands_actions,
                "market": market_actions,
            })
        if purpose == "dig":

            return save_action(obs, {
            "farmer": ["DIG"],
            "hands": hands_actions,
            "market": market_actions,
            })


    # ---------------------------------------------------------
    # FALLBACK
    # ---------------------------------------------------------

    '''if obs["hour"] == 0:

        return save_action(obs, {
            "farmer": ["WATER"],
            "hands": [hand_action] if hand_action else [],
            "market": market_actions,
        })'''

    return save_action(obs, {
        "farmer": ["PASS"],
        "hands": hands_actions,
        "market": market_actions,
    })

In [ ]:
for x in plant_debug:
    for p in x["plants"]:
        if p["yield"] >2:
            print(
                x["step"],
                x["day"],
                x["hour"],
                p
            )

In [ ]:
max_yield_seen = {}

for x in plant_debug:
    for p in x["plants"]:

        plant_id = (
            p["pos"],
            p["planted_day"]
        )

        y = p["yield"]

        if plant_id not in max_yield_seen:
            max_yield_seen[plant_id] = y
        else:
            max_yield_seen[plant_id] = max(
                max_yield_seen[plant_id],
                y
            )

from collections import Counter

print(Counter(max_yield_seen.values()))
#Counter({6: 34, 4: 6, 5: 4, 1: 1})
#Counter({6: 37, 4: 5, 5: 3, 3: 1})


In [ ]:
from collections import defaultdict

history = defaultdict(list)

for registro in plant_debug:
    for planta in registro["plants"]:
        plant_id = (
            planta["pos"],
            planta["planted_day"],
        )

        history[plant_id].append({
            "day": registro["day"],
            "hour": registro["hour"],
            "yield": planta["yield"],
        })

for plant_id, registros in history.items():
    max_yield = max(item["yield"] for item in registros)

    if max_yield < 6:
        ultimo = registros[-1]
        print(
            "planta:", plant_id,
            "| max_yield:", max_yield,
            "| último registro:", ultimo,
            "| idade final:", ultimo["day"] - plant_id[1],
        )

In [ ]:
for plant_id, history in yield_history.items():
    max_yield = max(y for step, y in history)

    if max_yield == 1:
        print(
            plant_id,
            "max =", max_yield,
            "last =", history[-1]
        )

In [ ]:
from collections import Counter, defaultdict

yield_history = defaultdict(list)

for x in plant_debug:
    for p in x["plants"]:

        plant_id = (
            p["pos"],
            p["planted_day"]
        )

        yield_history[plant_id].append(
            (x["step"], p["yield"])
        )

for plant_id, history in yield_history.items():
    max_yield = max(y for step, y in history)

    print(
        plant_id,
        "max =", max_yield,
        "last =", history[-1]
    )

In [ ]:
for x in action_debug:
    if 0 <= x["step"] <= 720:
        print(
            f"Step {x['step']} | "
            f"Day {x['day']} Hour {x['hour']} | "
            f"Farmer: {x.get('farmer')} @ {x['farmer_pos']} | "
            f"Hand: {x['hands']}"
        )

In [ ]:
print("Quantidade de chamadas do agent:", len(money_debug))
print("Último debug:", money_debug[-1])

print("Quantidade de steps:", len(env.steps))
print("Estado final:", env.steps[-1][0].status)
print("Detalhes finais:", env.steps[-1][0])

In [ ]:
for x in money_debug:
    if x["step"] == 0 or x["money"] != money_debug[x["step"] - 1]["money"]:
        print(x)

In [ ]:
for i in range(1, len(money_debug)):
    anterior = money_debug[i - 1]
    atual = money_debug[i]

    if atual["seed"] != anterior["seed"]:
        print(
            "SEED MUDOU:",
            anterior["step"], anterior["seed"],
            "→",
            atual["step"], atual["seed"]
        )

In [ ]:
for x in sell_debug:
    if x["shed"] != 0 or x["inventory"] != sell_debug[x["shed"] - 1]["inventory"]:
        print(x)

In [ ]:
for i in range(2, 10):
    obs = env.steps[i][0]["observation"]
    farm = obs["farms"][0]

    plants = []

    for y, row in enumerate(farm["tiles"]):
        for x, tile in enumerate(row):
            if (
                isinstance(tile, dict)
                and tile.get("kind") == "PLANT"
            ):
                plants.append(
                    (x, y, tile["watered_today"])
                )

    print(
        f"Step {i} | "
        f"Farmer: {farm['farmer']} | "
        f"Plants: {plants}"
    )

In [ ]:

env = make(
    "kaggriculture",
    configuration={
        "episodeSteps": 720
    }, debug =True
)

In [ ]:
observations = []
money_debug = []
sell_debug = []
market_debug = []
action_debug = []
plant_debug = []
env.run([agent, "random"])

In [ ]:
debug_targets = []



In [ ]:
def summarize_observation(obs):
    player = obs["player"]
    farm = obs["farms"][player]
    private = obs["private"]
    market = obs["market"]
    town = obs["town"]

    return {
        "day": obs["day"],
        "hour": obs["hour"],
        "money": farm["money"],
        "farmer_position": farm["farmer"],
        "hands": len(farm["hands"]),
        "unlocked_quadrants": farm["unlocked_quadrants"],
        "seeds": private["seeds"],
        "shed": private["shed"],
        "market_prices": market["prices"],
        "unlocked_shops": town["unlocked_shops"],
    }

In [ ]:
summary = summarize_observation(observations[1])
summary

In [ ]:
import pandas as pd
summaries = [
    summarize_observation(obs)
    for obs in observations
]

df = pd.DataFrame(summaries)

df

In [ ]:
def get_farmer_tile(obs):
    player = obs["player"]
    farm = obs["farms"][player]

    x, y = farm["farmer"]

    return farm["tiles"][y][x]

In [ ]:
def print_daily_crop_history(observations):
    for obs in observations:
        if obs["hour"] != 0:
            continue

        player = obs["player"]
        farm = obs["farms"][player]
        tile = get_farmer_tile(obs)

        crop = tile.get("crop") if isinstance(tile, dict) else None

        print(
            f"Day {obs['day']} | "
            f"Money: {farm['money']:.0f} | "
            f"Crop: {crop} | "
            f"Age: {obs['day'] - tile['planted_day'] if crop else '-'} | "
            f"Yield: {tile['yield_units'] if crop else '-'} | "
            f"Price: {obs['market']['prices'].get(crop, '-') if crop else '-'}"
        )

In [ ]:
print_daily_crop_history(observations)

In [ ]:
for obs in observations:

    farm = obs["farms"][obs["player"]]

    for y, row in enumerate(farm["tiles"]):
        for x, tile in enumerate(row):

            if not isinstance(tile, dict):
                continue

            if tile.get("kind") != "PLANT":
                continue

            print(
                f"Step {obs['step']} | "
                f"Day {obs['day']} Hour {obs['hour']} | "
                f"Pos {(x, y)} | "
                f"Planted {tile['planted_day']} | "
                f"Yield {tile['yield_units']} | "
                f"Watered {tile['watered_today']} | "
                f"Unwatered {tile['consecutive_unwatered']} | "
                f"Decay {tile['max_lifespan_step'] - obs['step']}"
            )

In [ ]:
for step_index in range(len(env.steps)):
    for player_state in env.steps[step_index]:
        if player_state.observation["day"] == 5 and player_state.observation["hour"] in [0, 1, 2,3,4,5]:
            print(
                f"Step: {step_index} | "
                f"Day: {player_state.observation['day']} | "
                f"Hour: {player_state.observation['hour']} | "
                f"Action: {player_state.action}"
            )

In [ ]:
obs_final = env_test.steps[-1][0]["observation"]

print("Dinheiro final:", obs_final["farms"][0]["money"])
print("Shed:", obs_final["private"]["shed"])
print("Seeds:", obs_final["private"]["seeds"])
print("Inventory:", obs_final["private"]["inventories"][0])

In [ ]:
print("Quantidade de registros:", len(sell_debug))

for x in sell_debug:
    if x["shed"] > 0:
        print(x)

In [ ]:
for step_index in range(118, 126):
    for player_state in env.steps[step_index]:
        obs = player_state.observation

        if obs["player"] == 0:
            print(
                f"Step {step_index} | "
                f"Day {obs['day']} | "
                f"Hour {obs['hour']} | "
                f"Position {obs['farms'][0]['farmer']} | "
                f"Action {player_state.action} | "
                f"Inventory {obs['private']['inventories'][0]}"
            )

In [ ]:
for i, obs in enumerate(observations):
    if obs["day"] <= 4:
        print(
            f"Step {obs['step']} | "
            f"Day {obs['day']} | "
            f"Hour {obs['hour']} | "
            f"Seeds: {obs['private']['seeds'].get(CROP, 0)} | "
            f"Inventory: {obs['private']['inventories'][0]} | "
            f"Shed: {obs['private']['shed'].get(CROP, 0)}"
        )

## What's wrong with this agent?

The melon agent has a few problems:
  - It never hires farm hands or buys more land, so it's stuck with one farmer working a single quadrant.     
  - It only grows melons, so when other crops are fetching a higher price it has nothing to sell.
  - It never fertilizes its plants, leaving a yield bonus on the table.                           
  - When it does sell, it dumps the entire inventory in one order, which can crash the melon price below the       
  threshold partway through the sale.                                                                         
                                      
Now it's your turn to make improvements!

## Making a submission

You can either submit a main.py, a tar.gz (or zip) with a main.py in it, or submit a notebook with a main.py or submission.tar.gz

There are three ways to subit.
1. using the [Submit Agent](https://www.kaggle.com/competitions/kaggriculture-gdm-internal) button on the homepage and uploading the file
2. using the Kaggle CLI (as described in agents.py in the competition dataset)
3. submitting a notebook with a submission.py or submission.tar.gz

## Submit to competition

Now that we have a main.py, all you need to do is click "Submit to competition" on the right and watch your entry show up on the competition leaderboard! Best of luck!